# SynthDet — Pipeline Demo

This notebook demonstrates the **end-to-end pipeline**: analyze a YOLO dataset, generate targeted synthetic data to fill distribution gaps, and measure the impact on model performance.

**Steps:**
1. Load & analyze dataset — class distribution, size buckets, spatial coverage
2. Configure the pipeline — generation methods, prompts, parameters
3. Dry-run — preview the synthesis strategy and estimate API costs
4. Run the pipeline — generate, split, validate
5. Verify synthetic data — CLIP quality filtering + diversity analysis
6. Inspect generated samples — visualize images with bboxes
7. Compare distributions — before/after size, spatial, and per-class balance
8. Train & compare — baseline (original only) vs augmented (original + synthetic)

**Generation methods:** Compositor (offline) | Generative Compositor (API key) | Inpainting (Vertex AI) | Modify-and-Annotate (Vertex AI)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import sys, os, random, warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
def resize_if_needed(img, max_size):
    h, w = img.shape[:2]
    if max(h, w) <= max_size:
        return img
    scale = max_size / max(h, w)
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

PROJECT_ROOT = './laptop defects all sides.v1-v1-raw.yolov12/'
sys.path.insert(0, '../')
Path("..").resolve()

#from synthdet.utils.image import resize_if_needed

# ── Paths ────────────────────────────────────────────────────────────────
# Point DATA_YAML at your YOLO dataset's data.yaml.
# Default: the 1-class Scratch dataset bundled in data/.
DATA_YAML = PROJECT_ROOT + "data.yaml"

OUTPUT_DIR = Path("demo_pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_IMAGE_SIZE = 1280
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Visualization helpers ────────────────────────────────────────────────
COLORS = [
    "#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6",
    "#1abc9c", "#e67e22", "#2980b9", "#c0392b", "#27ae60",
]

def show_bboxes(img_rgb, bboxes, class_names=None, ax=None, title=None):
    """Draw bboxes with class labels on an image."""
    if ax is None:
        _, ax = plt.subplots()
    h, w = img_rgb.shape[:2]
    ax.imshow(img_rgb)
    for b in bboxes:
        x1 = (b.x_center - b.width / 2) * w
        y1 = (b.y_center - b.height / 2) * h
        bw, bh = b.width * w, b.height * h
        color = COLORS[b.class_id % len(COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), bw, bh, linewidth=2,
            edgecolor=color, facecolor="none",
            boxstyle="round,pad=0",
        )
        ax.add_patch(rect)
        if class_names and b.class_id < len(class_names):
            label = class_names[b.class_id]
        else:
            label = str(b.class_id)
        ax.text(
            x1, y1 - 4, label, fontsize=6, color="white",
            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"),
        )
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

def to_rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def pct_bar(ax, labels, values, colors, title, horizontal=True):
    """Bar chart showing percentages (values should already be %)."""
    if horizontal:
        bars = ax.barh(labels, values, color=colors, edgecolor="white")
        for bar, v in zip(bars, values):
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{v:.1f}%", va="center", fontsize=9)
        ax.set_xlabel("%")
        ax.invert_yaxis()
    else:
        bars = ax.bar(labels, values, color=colors, edgecolor="white")
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    f"{v:.1f}%", ha="center", fontsize=8)
        ax.set_ylabel("%")
    ax.set_title(title, fontweight="bold")

print(f"Dataset:   {DATA_YAML}")
print(f"Output:    {OUTPUT_DIR.resolve()}")
print(f"Image cap: {MAX_IMAGE_SIZE}px")
print("Setup complete.")

---
## Step 1: Load & Analyze Dataset

Parse the YOLO dataset and compute statistics — class distribution, size buckets, spatial coverage, and gaps.

In [ ]:
from synthdet.analysis.loader import load_yolo_dataset
from synthdet.analysis.statistics import compute_dataset_statistics
from synthdet.config import SynthDetConfig
from synthdet.types import BBoxSizeBucket, SpatialRegion

dataset = load_yolo_dataset(DATA_YAML)
base_config = SynthDetConfig(max_image_size=MAX_IMAGE_SIZE)
stats = compute_dataset_statistics(dataset, base_config.analysis)

print(f"Classes ({len(dataset.class_names)}): {dataset.class_names}")
print(f"Train: {len(dataset.train)} | Valid: {len(dataset.valid)} | Test: {len(dataset.test)}")
print(f"Total annotations: {stats.total_annotations}")
print(f"Negative images:   {sum(1 for r in dataset.train if r.is_negative)}")
print(f"Bucket uniformity: {stats.bucket_uniformity:.3f}")
print(f"Spatial uniformity: {stats.region_uniformity:.3f}")

# ── 3-panel overview: class counts | size buckets | spatial heatmap ───
BUCKET_ORDER = ["tiny", "small", "medium", "large"]
BUCKET_COLORS = {"tiny": "#e74c3c", "small": "#f39c12", "medium": "#3498db", "large": "#2ecc71"}
REGION_ORDER = [
    "top_left", "top_center", "top_right",
    "middle_left", "middle_center", "middle_right",
    "bottom_left", "bottom_center", "bottom_right",
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: annotations per class
class_counts = {dataset.class_names[cd.class_id]: cd.count for cd in stats.class_distributions}
names = sorted(class_counts, key=class_counts.get)
counts = [class_counts[n] for n in names]
colors = [COLORS[dataset.class_names.index(n) % len(COLORS)] for n in names]
bars = axes[0].barh(names, counts, color=colors, edgecolor="white")
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 str(c), va="center", fontsize=9)
axes[0].set_title("Annotations per Class", fontweight="bold")
axes[0].set_xlabel("Count")

# Panel 2: size bucket distribution (%)
total_ann = max(stats.total_annotations, 1)
bucket_pcts = [stats.overall_bucket_counts.get(BBoxSizeBucket(b), 0) / total_ann * 100
               for b in BUCKET_ORDER]
pct_bar(axes[1], BUCKET_ORDER, bucket_pcts,
        [BUCKET_COLORS[b] for b in BUCKET_ORDER], "Size Bucket Distribution")

# Panel 3: spatial heatmap
grid = np.zeros((3, 3))
for i, reg in enumerate(REGION_ORDER):
    r, c = divmod(i, 3)
    grid[r, c] = stats.overall_region_counts.get(SpatialRegion(reg), 0)
im = axes[2].imshow(grid, cmap="YlOrRd")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, str(int(grid[i, j])), ha="center", va="center",
                     fontsize=12, fontweight="bold")
axes[2].set_xticks([0, 1, 2]); axes[2].set_xticklabels(["Left", "Center", "Right"])
axes[2].set_yticks([0, 1, 2]); axes[2].set_yticklabels(["Top", "Middle", "Bottom"])
axes[2].set_title("Spatial Distribution", fontweight="bold")
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

---
## Step 2: Define Custom Prompts

SynthDet uses a **prompt hierarchy** for API-based generation methods:

```
class_prompts (per-class, highest priority)
  -> default_prompts (fallback)
    -> task.suggested_prompts (auto-generated from class names)
      -> prompt_template (wraps the chosen prompt with context)
```

We define custom prompts for each generation method:

- **Inpainting** (`InpaintingConfig.class_prompts`): paints defects into masked regions
- **Generative Compositor** (`GenerativeDefectCompositor`): generates isolated defect patches, then Poisson-blends them onto backgrounds
- **Modify-and-Annotate** (`ModifyAnnotateConfig.class_prompts`): transforms whole images to add damage, then auto-annotates with Grounding DINO
- **Compositor** (`CompositorConfig`): offline Poisson blending of real patches — no prompts needed, but we tune blending parameters

In [ ]:
from synthdet.config import (
    SynthDetConfig,
    AnalysisConfig,
    CompositorConfig,
    InpaintingConfig,
    ModifyAnnotateConfig,
    AugmentationConfig,
    TrainingConfig,
)
from synthdet.pipeline.config_schema import PipelineConfig

# ── Custom prompts for Inpainting ────────────────────────────────────
# These are used by Imagen 3 to paint defects into masked regions.
# Each defect gets a prompt randomly sampled from its class list.
INPAINTING_PROMPTS = {
    "hard scratch": [
        "A deep scratch gouged into a laptop lid, with visible groove depth",
        "Multiple deep parallel scratches on a laptop lid, harsh lighting",
    ],
    "minor scratch": [
        "A light surface scratch on a laptop lid, barely catching the light",
        "A thin cosmetic scratch on a laptop lid",
    ],
    "sticker marks": [
        "Ligth adhesive residue marks on a smooth laptop surface",
        "Sticker removal residue with partial paper remnants on a flat laptop lid",
    ],
    "broken": [
        "A cracked section of laptop lid",
        "A damaged region of a laptop lid",
    ],
    "minor dent": [
        "A small dent on a flat laptop lid, with slight shadow",
    ],
    "minor crack": [
        "A thin hairline crack on a laptop surface, barely visible but distinct",
    ],
}

MODIFY_PROMPTS = {
    "hard scratch": [
        "Add deep, clearly visible white scratch lines gouged into this laptop surface",
    ],
    "minor scratch": [
        "Add clearly visible light scratch lines on this laptop top cover surface",
    ],
    "sticker marks": [
        "Add a clearly visible rectangular patch of sticky adhesive residue on this laptop",
    ],
    "broken": [
        "Add broken area with on this laptop lid",
    ],
    "minor dent": [
        "Add a clearly visible shallow dent creating a noticeable shadow on this laptop",
    ],
    "minor crack": [
        "Add a clearly visible thin dark crack line running across this laptop surface",
    ],
}

INPAINTING_TEMPLATE = (
    "{prompt}. The {class_name} should appear directly on the surface of "
    "the laptop lid, matching the existing material and lighting. "
    "Keep the rest of the image unchanged. Photorealistic, top-down view."
)

# ── Build the full pipeline config ───────────────────────────────────
synthdet_cfg = SynthDetConfig(
    max_image_size=MAX_IMAGE_SIZE,
    analysis=AnalysisConfig(
        multiplier=1.5,            # was 1.0 — generate fewer, higher-quality images
        negative_ratio=0.10,       # was 0.15 — fewer negatives
        min_per_bucket=30,         # was 50 — less aggressive gap filling
    ),
    compositor=CompositorConfig(
        blend_mode="mixed",
        max_defects_per_image=10,   # was 3 — fewer defects = more realistic
        rotation_jitter=10.0,      # was 15 — less extreme rotation
        scale_jitter=(0.6, 1.0),   # was (0.7, 1.3) — never upscale patches
        valid_zone_margin=0.05,
    ),

    augmentation=AugmentationConfig(
        variants_per_image=3,
        horizontal_flip_p=0.5,
        brightness_contrast_p=0.3,
    ),
)

pipeline_config = PipelineConfig(
    synthdet=synthdet_cfg,
    methods=["compositor", "inpainting", "generative_compositor"],
    train_split_ratio=0.85,
    validate_output=True,
    augment=True,  # skip augmentation to keep dataset manageable for training
)

# ── Print key config fields ──────────────────────────────────────────
print("Pipeline Configuration")
print("=" * 50)
print(f"Methods:            {pipeline_config.methods}")
print(f"Multiplier:         {synthdet_cfg.analysis.multiplier}x")
print(f"Negative ratio:     {synthdet_cfg.analysis.negative_ratio}")
print(f"Max image size:     {synthdet_cfg.max_image_size}px")
print(f"Augmentation:       {pipeline_config.augment}")
print(f"Validation:         {pipeline_config.validate_output}")
print()
print("Custom Inpainting Prompts:")
for cls, prompts in INPAINTING_PROMPTS.items():
    print(f"  {cls}: {len(prompts)} prompt(s)")
print()
print("Custom Modify-and-Annotate Prompts:")
for cls, prompts in MODIFY_PROMPTS.items():
    print(f"  {cls}: {len(prompts)} prompt(s)")

---
## Step 3: Dry Run — Cost Estimate

Before calling any APIs, compute the **synthesis strategy** to see what the pipeline will generate.
This shows the planned image count per method and estimated API costs — without making any API calls.

In [ ]:
from synthdet.analysis.strategy import generate_synthesis_strategy

strategy = generate_synthesis_strategy(dataset, stats, synthdet_cfg.analysis)

total_to_generate = sum(t.num_images for t in strategy.generation_tasks)
num_methods = len(pipeline_config.methods)
cost_per_api_image = synthdet_cfg.inpainting.cost_per_image

print("Synthesis Strategy")
print("=" * 50)
print(f"Target dataset size: {strategy.target_total_images} images")
print(f"Images to generate:  {total_to_generate}")
print(f"Negative ratio:      {strategy.negative_ratio:.0%}")
print(f"Generation tasks:    {len(strategy.generation_tasks)}")
print()

# ── Cost estimate per method ─────────────────────────────────────────
images_per_method = total_to_generate // num_methods
api_methods = [m for m in pipeline_config.methods if m != "compositor"]

print(f"Scaling across {num_methods} methods (~{images_per_method} images each):")
print(f"  {'compositor':25s}  ~{images_per_method:4d} images   $0.00 (offline)")
for m in api_methods:
    est_cost = images_per_method * cost_per_api_image
    print(f"  {m:25s}  ~{images_per_method:4d} images   ${est_cost:.2f}")

total_api_cost = images_per_method * cost_per_api_image * len(api_methods)
print(f"  {'TOTAL':25s}  ~{total_to_generate:4d} images   ${total_api_cost:.2f}")

# ── Visualize generation tasks by priority ───────────────────────────
tasks = strategy.generation_tasks
if tasks:
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(tasks) * 0.5)))

    # Left: images per task with priority coloring
    rationales = [t.rationale[:50] + "..." if len(t.rationale) > 50 else t.rationale
                  for t in tasks]
    img_counts = [t.num_images for t in tasks]
    priorities = [t.priority for t in tasks]
    bar_colors = plt.cm.RdYlGn([p for p in priorities])  # red=low, green=high

    axes[0].barh(range(len(tasks)), img_counts, color=bar_colors, edgecolor="white")
    axes[0].set_yticks(range(len(tasks)))
    axes[0].set_yticklabels(rationales, fontsize=8)
    for i, (cnt, pri) in enumerate(zip(img_counts, priorities)):
        axes[0].text(cnt + 1, i, f"{cnt} imgs (p={pri:.1f})", va="center", fontsize=8)
    axes[0].set_title("Generation Tasks (by rationale)", fontweight="bold")
    axes[0].set_xlabel("Images to generate")
    axes[0].invert_yaxis()

    # Right: image allocation across methods
    method_names = pipeline_config.methods
    method_imgs = [images_per_method] * len(method_names)
    method_costs = [0.0 if m == "compositor" else images_per_method * cost_per_api_image
                    for m in method_names]
    method_colors = ["#2ecc71" if m == "compositor" else "#3498db" for m in method_names]

    bars = axes[1].bar(method_names, method_imgs, color=method_colors, edgecolor="white")
    for bar, cost in zip(bars, method_costs):
        label = "offline" if cost == 0 else f"${cost:.2f}"
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                     label, ha="center", fontsize=9)
    axes[1].set_title("Images per Method", fontweight="bold")
    axes[1].set_ylabel("Images")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

---
## Step 4: Run the Pipeline

The pipeline executes these steps automatically:

1. **Analyze** the dataset to identify distribution gaps
2. **Generate** synthesis tasks with priorities (size bucket gaps, spatial gaps, negatives)
3. **Scale** tasks across methods (each method gets `N / num_methods` images)
4. **Run** each generation method with custom prompts
5. **Augment** with classical transforms (flips, color jitter, noise)
6. **Split** into train/valid
7. **Validate** output dataset integrity

> **Auth levels:**
> - No API key → **compositor only** (fully offline)
> - `GOOGLE_API_KEY` → compositor + **generative compositor** (Gemini API)
> - `GOOGLE_CLOUD_PROJECT` → all of the above + **inpainting** + **modify-and-annotate** (Vertex AI)

In [ ]:
import gc
from synthdet.pipeline.orchestrator import run_pipeline

# ── Select methods based on available credentials ────────────────────
# Auth requirements:
#   compositor             — offline, no credentials needed
#   generative_compositor  — GOOGLE_API_KEY (Gemini API) or Vertex AI
#   inpainting             — Vertex AI only (edit_image is Vertex-only)
#   modify_annotate        — Vertex AI only (controlled editing is Vertex-only)

has_api_key = bool(os.environ.get("GOOGLE_API_KEY"))
has_vertex = bool(os.environ.get("GOOGLE_CLOUD_PROJECT"))
has_api_key

In [ ]:

if has_vertex:
    run_methods = ["compositor", "inpainting", "generative_compositor"]
    print("GOOGLE_CLOUD_PROJECT detected — running all 4 methods (Vertex AI)")
elif has_api_key:
    run_methods = ["compositor", "generative_compositor"]
    print("GOOGLE_API_KEY detected — running compositor + generative compositor")
    print("(inpainting & modify_annotate require Vertex AI)")
else:
    run_methods = ["compositor"]
    print("No API credentials — running compositor only (offline)")
    print("Set GOOGLE_API_KEY to also enable generative compositor.")
    print("Set GOOGLE_CLOUD_PROJECT to enable all 4 methods.")

run_config = pipeline_config.model_copy(update={"methods": run_methods})

print(f"\nRunning with methods: {run_config.methods}")
print("-" * 50)


In [ ]:
# ── Run ───────────────────────────────────────────────────────────────
pipeline_output = OUTPUT_DIR / "pipeline_run"

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, force=True)
result = run_pipeline(
    data_yaml=DATA_YAML,
    output_dir=pipeline_output,
    config=run_config,
    seed=SEED,
)

gc.collect()

# ── Summary ───────────────────────────────────────────────────────────
print("\nPipeline Result")
print("=" * 50)
print(f"Methods used:    {result.methods_used}")
for method, count in result.records_per_method.items():
    cost = result.cost_per_method.get(method, 0)
    print(f"  {method:25s}  {count:4d} images   ${cost:.2f}")
print(f"Total records:   {result.total_records}")
print(f"Train / Valid:   {result.train_count} / {result.valid_count}")
print(f"Total cost:      ${result.total_cost_usd:.2f}")
if result.validation_report:
    vr = result.validation_report
    status = "PASSED" if vr.is_valid else f"FAILED ({len(vr.errors)} errors)"
    print(f"Validation:      {status}")

---
## Step 5: Verify Synthetic Data Quality

Before using synthetic data for training, we run two quality checks:

1. **CLIP Verifier** — crop each bbox, score it against its class label using CLIP cosine similarity. Images where *all* bboxes score below the threshold are removed (they likely contain failed generations). Mixed-quality images keep only the passing bboxes.

2. **Diversity Analyzer** — compute OpenCLIP embeddings for all synthetic images and measure pairwise cosine distance, per-class diversity, coverage of the original dataset, and outlier detection.

This step modifies the generated dataset on disk before training.

In [ ]:
print(1)

In [ ]:
from synthdet.analysis.loader import load_yolo_dataset as load_output
from synthdet.annotate.verifier import CLIPVerifier
from synthdet.types import BBox

# ── Load the generated dataset ───────────────────────────────────────
output_dataset = load_output(pipeline_output / "data.yaml")
class_names = output_dataset.class_names

print(f"Generated dataset: {len(output_dataset.train)} train, {len(output_dataset.valid)} valid")
print(f"Classes: {class_names}")

# ── CLIP Verification ────────────────────────────────────────────────
CLIP_THRESHOLD = 0.25

verifier = CLIPVerifier(
    model="openai/clip-vit-base-patch32",
    min_confidence=CLIP_THRESHOLD,
    class_names=class_names,
)

all_records = output_dataset.train + output_dataset.valid
all_scores = []       # (record_idx, bbox_idx, score)
per_image_scores = [] # (record_idx, mean_score, min_score, n_bboxes, n_pass)

for rec_idx, rec in enumerate(all_records):
    if rec.is_negative:
        # Negative images have no bboxes to verify — keep them
        per_image_scores.append((rec_idx, 1.0, 1.0, 0, 0))
        continue

    img = rec.load_image()
    results = verifier.verify(img, rec.bboxes)
    rec.image = None

    scores = [s for _, s in results]
    passing = sum(1 for s in scores if s >= CLIP_THRESHOLD)
    all_scores.extend((rec_idx, bi, s) for bi, (_, s) in enumerate(results))
    per_image_scores.append((
        rec_idx,
        float(np.mean(scores)) if scores else 0.0,
        float(np.min(scores)) if scores else 0.0,
        len(scores),
        passing,
    ))

# ── Score distribution ───────────────────────────────────────────────
bbox_scores = [s for _, _, s in all_scores]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of per-bbox CLIP scores
if bbox_scores:
    axes[0].hist(bbox_scores, bins=30, color="#3498db", edgecolor="white", alpha=0.8)
    axes[0].axvline(CLIP_THRESHOLD, color="#e74c3c", ls="--", lw=2, label=f"Threshold ({CLIP_THRESHOLD})")
    n_pass = sum(1 for s in bbox_scores if s >= CLIP_THRESHOLD)
    n_fail = len(bbox_scores) - n_pass
    axes[0].set_title(f"Per-BBox CLIP Score ({n_pass} pass, {n_fail} fail)", fontweight="bold")
    axes[0].set_xlabel("CLIP Similarity Score")
    axes[0].set_ylabel("Count")
    axes[0].legend()

# Per-image mean score
mean_scores = [ms for _, ms, _, nb, _ in per_image_scores if nb > 0]
if mean_scores:
    axes[1].hist(mean_scores, bins=20, color="#2ecc71", edgecolor="white", alpha=0.8)
    axes[1].axvline(CLIP_THRESHOLD, color="#e74c3c", ls="--", lw=2, label=f"Threshold ({CLIP_THRESHOLD})")
    axes[1].set_title("Per-Image Mean CLIP Score", fontweight="bold")
    axes[1].set_xlabel("Mean CLIP Score")
    axes[1].set_ylabel("Count")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Identify images to remove / bboxes to filter ────────────────────
# An image is removed if ALL its bboxes fail the threshold.
# Otherwise, only failing bboxes are dropped from the label file.
images_to_remove = []
images_to_filter = []  # (rec_idx, list of passing bbox indices)

for rec_idx, mean_s, min_s, n_bboxes, n_pass in per_image_scores:
    if n_bboxes == 0:
        continue  # negative — keep
    if n_pass == 0:
        images_to_remove.append(rec_idx)
    elif n_pass < n_bboxes:
        # Keep image but only the passing bboxes
        passing_indices = [bi for ri, bi, s in all_scores
                          if ri == rec_idx and s >= CLIP_THRESHOLD]
        images_to_filter.append((rec_idx, passing_indices))

print(f"\nCLIP Verification Summary (threshold={CLIP_THRESHOLD}):")
print(f"  Total images checked:  {sum(1 for _, _, _, nb, _ in per_image_scores if nb > 0)}")
print(f"  Images to REMOVE:      {len(images_to_remove)} (all bboxes below threshold)")
print(f"  Images to FILTER:      {len(images_to_filter)} (some bboxes below threshold)")
print(f"  Images passing fully:  {sum(1 for _, _, _, nb, np_ in per_image_scores if nb > 0 and np_ == nb)}")

In [ ]:
# ── Show discarded images with bounding boxes ─────────────────────────
N_SHOW = 8
show_indices = images_to_remove[:N_SHOW]

if not show_indices:
    print("No images were discarded by CLIP verification.")
else:
    fig, axes = plt.subplots(1, len(show_indices), figsize=(5 * len(show_indices), 5))
    if len(show_indices) == 1:
        axes = [axes]

    colors = plt.cm.tab10.colors

    for ax, rec_idx in zip(axes, show_indices):
        rec = all_records[rec_idx]
        img = cv2.imread(str(rec.image_path))
        if img is None:
            ax.set_title("(file missing)"); ax.axis("off")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = resize_if_needed(img, MAX_IMAGE_SIZE)
        h, w = img.shape[:2]
        ax.imshow(img)

        # Draw bboxes with scores
        rec_scores = {bi: s for ri, bi, s in all_scores if ri == rec_idx}
        for bi, bbox in enumerate(rec.bboxes):
            cx, cy, bw, bh = bbox.x_center, bbox.y_center, bbox.width, bbox.height
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            bw_px = int(bw * w)
            bh_px = int(bh * h)
            color = colors[bbox.class_id % len(colors)]
            rect = mpatches.FancyBboxPatch(
                (x1, y1), bw_px, bh_px,
                linewidth=2, edgecolor=color, facecolor="none",
                boxstyle="round,pad=0",
            )
            ax.add_patch(rect)
            score = rec_scores.get(bi, 0.0)
            label = f"{class_names[bbox.class_id]} {score:.2f}"
            ax.text(x1, y1 - 4, label, fontsize=8, color="white",
                    bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))

        ax.set_title(f"Discarded #{rec_idx}", fontweight="bold")
        ax.axis("off")

    plt.suptitle(f"Discarded Images ({len(images_to_remove)} total, showing {len(show_indices)})",
                 fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Apply filtering on disk ───────────────────────────────────────────
removed_count = 0
filtered_bbox_count = 0

for rec_idx in images_to_remove:
    rec = all_records[rec_idx]
    img_path = rec.image_path
    lbl_path = img_path.parent.parent / "labels" / img_path.with_suffix(".txt").name
    if img_path.exists():
        img_path.unlink()
    if lbl_path.exists():
        lbl_path.unlink()
    removed_count += 1

for rec_idx, passing_indices in images_to_filter:
    rec = all_records[rec_idx]
    lbl_path = rec.image_path.parent.parent / "labels" / rec.image_path.with_suffix(".txt").name
    if not lbl_path.exists():
        continue
    lines = lbl_path.read_text().strip().splitlines()
    kept_lines = [lines[bi] for bi in passing_indices if bi < len(lines)]
    original_count = len(lines)
    lbl_path.write_text("\n".join(kept_lines) + ("\n" if kept_lines else ""))
    filtered_bbox_count += original_count - len(kept_lines)

print(f"Applied CLIP filtering to disk:")
print(f"  Images removed:     {removed_count}")
print(f"  BBoxes removed:     {filtered_bbox_count} (from partially passing images)")

# ── Reload the cleaned dataset ───────────────────────────────────────
output_dataset = load_output(pipeline_output / "data.yaml")
print(f"\nCleaned dataset: {len(output_dataset.train)} train, {len(output_dataset.valid)} valid")
print(f"Total annotations: {sum(len(r.bboxes) for r in output_dataset.train + output_dataset.valid)}")

gc.collect()

In [ ]:
# ── Diversity Analysis ────────────────────────────────────────────────
from synthdet.analysis.diversity import DiversityAnalyzer
analyzer = DiversityAnalyzer()
synth_report = analyzer.analyze(
    output_dataset,
    split="train",
    reference_dataset=dataset,
    reference_split="train",
)

orig_report = analyzer.analyze(
    dataset,
    split="train",
)

# Also analyze original dataset for comparison
orig_report = analyzer.analyze(
    dataset,
    split="train",
)

gc.collect()

# ── Diversity comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Overall diversity comparison
metric_names = ["Mean Pairwise\nCosine Distance"]
orig_vals = [orig_report.mean_pairwise_cosine_distance]
synth_vals = [synth_report.mean_pairwise_cosine_distance]
x = np.arange(len(metric_names))
w = 0.3
axes[0].bar(x - w/2, orig_vals, w, label="Original", color="#3498db", edgecolor="white")
axes[0].bar(x + w/2, synth_vals, w, label="Synthetic", color="#e74c3c", edgecolor="white")
for i, (ov, sv) in enumerate(zip(orig_vals, synth_vals)):
    axes[0].text(i - w/2, ov + 0.005, f"{ov:.3f}", ha="center", fontsize=9)
    axes[0].text(i + w/2, sv + 0.005, f"{sv:.3f}", ha="center", fontsize=9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].set_title("Overall Diversity (higher = more diverse)", fontweight="bold")
axes[0].legend()
axes[0].set_ylim(0, max(max(orig_vals), max(synth_vals)) * 1.3)

# Panel 2: Per-class diversity
all_classes_div = sorted(set(list(orig_report.per_class_diversity.keys())
                              + list(synth_report.per_class_diversity.keys())))
# Filter out _negative for cleaner chart
all_classes_div = [c for c in all_classes_div if c != "_negative"]
if all_classes_div:
    x = np.arange(len(all_classes_div))
    w = 0.3
    orig_class_vals = [orig_report.per_class_diversity.get(c, 0) for c in all_classes_div]
    synth_class_vals = [synth_report.per_class_diversity.get(c, 0) for c in all_classes_div]
    axes[1].bar(x - w/2, orig_class_vals, w, label="Original", color="#3498db", edgecolor="white")
    axes[1].bar(x + w/2, synth_class_vals, w, label="Synthetic", color="#e74c3c", edgecolor="white")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(all_classes_div, rotation=35, ha="right", fontsize=8)
    axes[1].set_ylabel("Mean Pairwise Cosine Distance")
    axes[1].set_title("Per-Class Diversity", fontweight="bold")
    axes[1].legend(fontsize=8)

# Panel 3: Summary text
coverage_str = f"{synth_report.coverage_ratio:.1%}" if synth_report.coverage_ratio is not None else "N/A"
info_text = (
    f"Diversity Analysis\n"
    f"{'=' * 35}\n"
    f"\n"
    f"Original Dataset\n"
    f"  Images:     {orig_report.num_images}\n"
    f"  Diversity:  {orig_report.mean_pairwise_cosine_distance:.3f}\n"
    f"  Outliers:   {len(orig_report.outlier_indices)}\n"
    f"\n"
    f"Synthetic Dataset (after CLIP filter)\n"
    f"  Images:     {synth_report.num_images}\n"
    f"  Diversity:  {synth_report.mean_pairwise_cosine_distance:.3f}\n"
    f"  Outliers:   {len(synth_report.outlier_indices)}\n"
    f"  Coverage:   {coverage_str}\n"
)
axes[2].text(0.05, 0.5, info_text, transform=axes[2].transAxes,
             fontsize=10, verticalalignment="center", fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1", edgecolor="#bdc3c7"))
axes[2].axis("off")
axes[2].set_title("Summary", fontweight="bold")

plt.tight_layout()
plt.show()

# ── Print outlier details ────────────────────────────────────────────
if synth_report.outlier_indices:
    print(f"\nSynthetic outliers ({len(synth_report.outlier_indices)} images with z-score > 2.5):")
    synth_records = output_dataset.train
    for idx, z in zip(synth_report.outlier_indices, synth_report.outlier_z_scores):
        if idx < len(synth_records):
            print(f"  [{idx}] z={z:.2f}  {synth_records[idx].image_path.name}")
else:
    print("\nNo outliers detected in synthetic dataset.")

---
## Step 6: Inspect Generated Samples

Visualize random positive samples (with bboxes and annotation source) and negative samples (clean images). The pie chart shows the breakdown of how annotations were produced.

In [ ]:
from collections import Counter

train_records = output_dataset.train

# ── Separate positive and negative samples ───────────────────────────
positives = [r for r in train_records if not r.is_negative]
negatives = [r for r in train_records if r.is_negative]
print(f"Generated: {len(positives)} positive, {len(negatives)} negative images")

# ── Positive sample grid ────────────────────────────────────────────
pos_samples = random.sample(positives, min(8, len(positives)))
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, rec in zip(axes.flat, pos_samples):
    img = resize_if_needed(rec.load_image(), MAX_IMAGE_SIZE)
    rec.image = None
    # Show annotation source in title
    srcs = set(b.source.value for b in rec.bboxes)
    src_str = "/".join(sorted(srcs))
    title = f"{len(rec.bboxes)} bbox ({src_str})"
    show_bboxes(to_rgb(img), rec.bboxes, output_dataset.class_names,
                ax=ax, title=title)
for ax in axes.flat[len(pos_samples):]:
    ax.axis("off")
fig.suptitle("Positive Samples (with annotation source)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Negative sample grid (if any) ───────────────────────────────────
if negatives:
    neg_samples = random.sample(negatives, min(4, len(negatives)))
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, rec in zip(axes, neg_samples):
        img = resize_if_needed(rec.load_image(), MAX_IMAGE_SIZE)
        rec.image = None
        ax.imshow(to_rgb(img))
        ax.set_title("negative (clean)", fontsize=10)
        ax.axis("off")
    for ax in axes[len(neg_samples):]:
        ax.axis("off")
    fig.suptitle("Negative Samples (no defects)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# ── Annotation source breakdown ─────────────────────────────────────
sources = Counter()
for rec in train_records:
    for b in rec.bboxes:
        sources[b.source.value] += 1
    if not rec.bboxes:
        sources["negative"] += 1

if sources:
    fig, ax = plt.subplots(figsize=(6, 6))
    labels = list(sources.keys())
    values = list(sources.values())
    ax.pie(values, labels=labels, autopct="%1.0f%%", startangle=90,
           colors=COLORS[:len(labels)])
    ax.set_title("Annotation Sources", fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Step 7: Before/After Comparison

Compare the **original** vs **synthetic** dataset distributions:
- **Size buckets** (%) — the pipeline targets underrepresented sizes
- **Spatial coverage** — shared colormap so heatmaps are directly comparable
- **Per-class balance** — grouped bars showing annotation counts before/after
- **BBox area histogram** — overlaid distributions of normalized bbox areas

> The synthetic dataset includes ~15% negative examples (clean images), critical for reducing false positives during training.

In [ ]:
after_stats = compute_dataset_statistics(output_dataset, base_config.analysis)

# ── Row 1: Size buckets (%) — before vs after ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

before_total = max(stats.total_annotations, 1)
after_total = max(after_stats.total_annotations, 1)

before_pcts = [stats.overall_bucket_counts.get(BBoxSizeBucket(b), 0) / before_total * 100
               for b in BUCKET_ORDER]
after_pcts = [after_stats.overall_bucket_counts.get(BBoxSizeBucket(b), 0) / after_total * 100
              for b in BUCKET_ORDER]

pct_bar(axes[0], BUCKET_ORDER, before_pcts,
        [BUCKET_COLORS[b] for b in BUCKET_ORDER], "Size Buckets: ORIGINAL (%)")
pct_bar(axes[1], BUCKET_ORDER, after_pcts,
        [BUCKET_COLORS[b] for b in BUCKET_ORDER], "Size Buckets: SYNTHETIC (%)")

# Match x-axis scale
x_max = max(max(before_pcts), max(after_pcts)) * 1.25
axes[0].set_xlim(0, x_max)
axes[1].set_xlim(0, x_max)

plt.tight_layout()
plt.show()

# ── Row 2: Spatial heatmaps with shared colormap ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

grid_before = np.zeros((3, 3))
grid_after = np.zeros((3, 3))
for i, reg in enumerate(REGION_ORDER):
    r, c = divmod(i, 3)
    grid_before[r, c] = stats.overall_region_counts.get(SpatialRegion(reg), 0)
    grid_after[r, c] = after_stats.overall_region_counts.get(SpatialRegion(reg), 0)

vmin = min(grid_before.min(), grid_after.min())
vmax = max(grid_before.max(), grid_after.max())

for ax, grid, title in [(axes[0], grid_before, "Spatial: ORIGINAL"),
                         (axes[1], grid_after, "Spatial: SYNTHETIC")]:
    im = ax.imshow(grid, cmap="YlOrRd", vmin=vmin, vmax=vmax)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(int(grid[i, j])), ha="center", va="center",
                    fontsize=12, fontweight="bold")
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["Left", "Center", "Right"])
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Top", "Middle", "Bottom"])
    ax.set_title(title, fontweight="bold")

fig.colorbar(im, ax=axes, shrink=0.8, label="Annotations")
plt.tight_layout()
plt.show()

# ── Row 3: Per-class balance (grouped bar chart) ────────────────────
before_class = {dataset.class_names[cd.class_id]: cd.count for cd in stats.class_distributions}
after_class = {output_dataset.class_names[cd.class_id]: cd.count for cd in after_stats.class_distributions}

all_classes = sorted(set(list(before_class.keys()) + list(after_class.keys())),
                     key=lambda n: before_class.get(n, 0), reverse=True)

x = np.arange(len(all_classes))
width = 0.35

fig, ax = plt.subplots(figsize=(max(10, len(all_classes) * 1.2), 5))
bars1 = ax.bar(x - width / 2, [before_class.get(c, 0) for c in all_classes],
               width, label="Original", color="#3498db", edgecolor="white")
bars2 = ax.bar(x + width / 2, [after_class.get(c, 0) for c in all_classes],
               width, label="Synthetic", color="#e74c3c", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(all_classes, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Annotations")
ax.set_title("Per-Class Annotation Count: Original vs Synthetic", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

# ── Row 4: BBox area distribution histogram ─────────────────────────
before_areas = [b.area for b in dataset.all_bboxes("train")]
after_areas = [b.area for b in output_dataset.all_bboxes("train")]

if before_areas and after_areas:
    fig, ax = plt.subplots(figsize=(10, 4))
    bins = np.linspace(0, max(max(before_areas), max(after_areas)) * 1.05, 40)
    ax.hist(before_areas, bins=bins, alpha=0.6, label="Original", color="#3498db", edgecolor="white")
    ax.hist(after_areas, bins=bins, alpha=0.6, label="Synthetic", color="#e74c3c", edgecolor="white")

    for thresh, label in [(0.005, "tiny/small"), (0.02, "small/med"), (0.08, "med/large")]:
        ax.axvline(thresh, color="gray", linestyle="--", alpha=0.6, lw=1)
        ax.text(thresh, ax.get_ylim()[1] * 0.92, f" {label}", fontsize=7, color="gray")

    ax.set_xlabel("Normalized BBox Area (w * h)")
    ax.set_ylabel("Count")
    ax.set_title("BBox Area Distribution: Original vs Synthetic", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()

# ── Summary metrics ──────────────────────────────────────────────────
print(f"Bucket uniformity:  {stats.bucket_uniformity:.3f} -> {after_stats.bucket_uniformity:.3f}")
print(f"Spatial uniformity: {stats.region_uniformity:.3f} -> {after_stats.region_uniformity:.3f}")
print(f"Total images:       {stats.total_images} -> {after_stats.total_images}")
print(f"Total annotations:  {stats.total_annotations} -> {after_stats.total_annotations}")

---
## Step 8: Train & Compare — Baseline vs Augmented

The key question: **does synthetic data actually help?**

We train two YOLOv8n models under identical conditions:
1. **Baseline** — trained on the **original** dataset only
2. **Augmented** — trained on **original + synthetic** data merged together

Both use the same hyperparameters, pretrained weights, and the **original validation set** (unchanged) so the comparison is fair. Training uses GPU for 50 epochs with early stopping (patience=10).

In [ ]:
import gc, shutil
gc.collect()

from synthdet.training.trainer import YOLOTrainer

TRAIN_EPOCHS = 50
TRAIN_DEVICE = "auto"  # uses GPU if available

# ── Shared training config ───────────────────────────────────────────
base_train_cfg = dict(
    model_arch="yolov8n.pt",
    epochs=TRAIN_EPOCHS,
    batch_size=8,
    imgsz=MAX_IMAGE_SIZE,
    patience=25,
    workers=4,
    device=TRAIN_DEVICE,
    project=str(OUTPUT_DIR / "runs"),
    pretrained=True,
)

# ── 7a: Train BASELINE on original data ──────────────────────────────
print("=" * 60)
print("  Training BASELINE model (original data only)")
print("=" * 60)

In [ ]:
baseline_cfg = TrainingConfig(**base_train_cfg, name="baseline")
baseline_trainer = YOLOTrainer(baseline_cfg)
baseline_result = baseline_trainer.train(data_yaml=DATA_YAML)

print(f"\nBaseline complete: {baseline_result.epochs_completed} epochs")
print(f"  mAP50:     {baseline_result.best_map50:.4f}")
print(f"  mAP50-95:  {baseline_result.best_map50_95:.4f}")
print(f"  Time:      {baseline_result.training_time_seconds:.0f}s")

gc.collect()

In [ ]:
# ── 7b: Merge original + synthetic into a combined dataset ───────────
# We combine the original train set with all synthetic images, but keep
# the original validation set unchanged so both models are evaluated on
# the exact same data.
import yaml as _yaml

merged_dir = OUTPUT_DIR / "merged_dataset"
merged_train_img = merged_dir / "train" / "images"
merged_train_lbl = merged_dir / "train" / "labels"
merged_val_img = merged_dir / "valid" / "images"
merged_val_lbl = merged_dir / "valid" / "labels"

for d in [merged_train_img, merged_train_lbl, merged_val_img, merged_val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

def _remap_labels(src_lbl: Path, dst_lbl: Path, nc: int):
    """Copy a YOLO label file, remapping out-of-range class IDs to 0.

    Auto-annotators (e.g. Grounding DINO) may produce class IDs that
    exceed the dataset's nc. For a 1-class dataset, every bbox is class 0.
    """
    if not src_lbl.is_file():
        dst_lbl.write_text("")
        return
    lines = src_lbl.read_text().strip().splitlines()
    fixed = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = int(parts[0])
            if cls_id >= nc:
                cls_id = 0  # remap to class 0
            fixed.append(f"{cls_id} " + " ".join(parts[1:]))
    dst_lbl.write_text("\n".join(fixed) + "\n" if fixed else "")

def _copy_split(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, prefix="", nc=1):
    """Copy images and labels, optionally prefixing filenames to avoid collisions."""
    count = 0
    src_img_dir, src_lbl_dir = Path(src_img_dir), Path(src_lbl_dir)
    if not src_img_dir.is_dir():
        return 0
    for img_path in sorted(src_img_dir.iterdir()):
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        dst_name = f"{prefix}{img_path.name}" if prefix else img_path.name
        shutil.copy2(img_path, dst_img_dir / dst_name)
        lbl_path = src_lbl_dir / img_path.with_suffix(".txt").name
        dst_lbl = dst_lbl_dir / Path(dst_name).with_suffix(".txt")
        _remap_labels(lbl_path, dst_lbl, nc)
        count += 1
    return count

nc = len(dataset.class_names)

# Resolve original dataset paths from ImageRecord paths (avoids data.yaml
# relative-path ambiguity).
orig_train_img = dataset.train[0].image_path.parent if dataset.train else None
orig_train_lbl = orig_train_img.parent / "labels" if orig_train_img else None
orig_val_img = dataset.valid[0].image_path.parent if dataset.valid else None
orig_val_lbl = orig_val_img.parent / "labels" if orig_val_img else None

# Copy original train
n_orig = _copy_split(orig_train_img, orig_train_lbl, merged_train_img, merged_train_lbl, nc=nc) if orig_train_img else 0

# Copy synthetic train (prefix "synth_" to avoid filename collisions)
synth_train_img = pipeline_output / "train" / "images"
synth_train_lbl = pipeline_output / "train" / "labels"
n_synth = _copy_split(synth_train_img, synth_train_lbl,
                       merged_train_img, merged_train_lbl, prefix="synth_", nc=nc)

# Copy original validation (unchanged)
n_val = _copy_split(orig_val_img, orig_val_lbl, merged_val_img, merged_val_lbl, nc=nc) if orig_val_img else 0

# Write merged data.yaml with RELATIVE paths (ultralytics resolves them
# relative to the yaml file's parent directory).
merged_data_yaml = merged_dir / "data.yaml"
merged_meta = {
    "train": "train/images",
    "val": "valid/images",
    "nc": nc,
    "names": dataset.class_names,
}
with open(merged_data_yaml, "w") as f:
    _yaml.dump(merged_meta, f)

print(f"Merged dataset: {merged_data_yaml}")
print(f"  Original train: {n_orig} images")
print(f"  Synthetic train: {n_synth} images")
print(f"  Combined train:  {n_orig + n_synth} images")
print(f"  Validation:      {n_val} images (original, unchanged)")
print(f"  Classes:         {nc} (out-of-range class IDs remapped to 0)")

In [ ]:
# ── 7c: Train AUGMENTED model on merged dataset ─────────────────────
print("=" * 60)
print("  Training AUGMENTED model (original + synthetic)")
print("=" * 60)
TRAIN_EPOCHS = 150
TRAIN_DEVICE = "auto"  # uses GPU if available

# ── Shared training config ───────────────────────────────────────────
base_train_cfg = dict(
    model_arch="yolov8s.pt",
    epochs=TRAIN_EPOCHS,
    batch_size=4,
    imgsz=MAX_IMAGE_SIZE,
    patience=50,
    workers=4,
    device=TRAIN_DEVICE,
    project=str(OUTPUT_DIR / "runs"),
    pretrained=True,
)
augmented_cfg = TrainingConfig(**base_train_cfg, name="augmented")
augmented_trainer = YOLOTrainer(augmented_cfg)
augmented_result = augmented_trainer.train(data_yaml=merged_data_yaml)

print(f"\nAugmented complete: {augmented_result.epochs_completed} epochs")
print(f"  mAP50:     {augmented_result.best_map50:.4f}")
print(f"  mAP50-95:  {augmented_result.best_map50_95:.4f}")
print(f"  Time:      {augmented_result.training_time_seconds:.0f}s")

gc.collect()

In [ ]:
# ── 7d: Side-by-side comparison ──────────────────────────────────────

def _extract_curve(result, key):
    """Extract a metric curve from training history, trying common column names."""
    if not result.metrics_history:
        return [], []
    col_map = {
        "box_loss": ["train/box_loss", "box_loss"],
        "mAP50": ["metrics/mAP50(B)", "mAP50"],
        "mAP50-95": ["metrics/mAP50-95(B)", "mAP50-95"],
    }
    candidates = col_map.get(key, [key])
    epochs, values = [], []
    for i, row in enumerate(result.metrics_history):
        for col in candidates:
            if col in row:
                epochs.append(i + 1)
                values.append(row[col])
                break
    return epochs, values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: mAP50 training curves
e_b, m_b = _extract_curve(baseline_result, "mAP50")
e_a, m_a = _extract_curve(augmented_result, "mAP50")
if e_b:
    axes[0].plot(e_b, m_b, "C0-", lw=2, label="Baseline")
if e_a:
    axes[0].plot(e_a, m_a, "C3-", lw=2, label="Augmented")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mAP50")
axes[0].set_ylim(0, 1)
axes[0].set_title("mAP50 over Training", fontweight="bold")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: Box loss curves
e_b, l_b = _extract_curve(baseline_result, "box_loss")
e_a, l_a = _extract_curve(augmented_result, "box_loss")
if e_b:
    axes[1].plot(e_b, l_b, "C0-", lw=2, label="Baseline")
if e_a:
    axes[1].plot(e_a, l_a, "C3-", lw=2, label="Augmented")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Box Loss")
axes[1].set_title("Box Loss over Training", fontweight="bold")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Panel 3: Final metrics comparison
metrics_names = ["mAP50", "mAP50-95"]
baseline_vals = [baseline_result.best_map50, baseline_result.best_map50_95]
augmented_vals = [augmented_result.best_map50, augmented_result.best_map50_95]

x = np.arange(len(metrics_names))
w = 0.3
bars1 = axes[2].bar(x - w/2, baseline_vals, w, label="Baseline", color="#95a5a6", edgecolor="white")
bars2 = axes[2].bar(x + w/2, augmented_vals, w, label="Augmented", color="#2ecc71", edgecolor="white")
for i, (bv, av) in enumerate(zip(baseline_vals, augmented_vals)):
    axes[2].text(i - w/2, bv + 0.01, f"{bv:.3f}", ha="center", fontsize=9)
    axes[2].text(i + w/2, av + 0.01, f"{av:.3f}", ha="center", fontsize=9)
    # Delta annotation
    delta = av - bv
    sign = "+" if delta >= 0 else ""
    axes[2].annotate(
        f"{sign}{delta:.3f}", xy=(i + w/2, av + 0.04),
        fontsize=10, fontweight="bold", ha="center",
        color="#27ae60" if delta > 0 else "#e74c3c",
    )
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].set_ylim(0, 1)
axes[2].set_title("Best Metrics: Baseline vs Augmented", fontweight="bold")
axes[2].legend()

plt.tight_layout()
plt.show()

# ── Print summary ────────────────────────────────────────────────────
delta_map50 = augmented_result.best_map50 - baseline_result.best_map50
delta_map50_95 = augmented_result.best_map50_95 - baseline_result.best_map50_95
print(f"Baseline  -> mAP50: {baseline_result.best_map50:.4f}  |  mAP50-95: {baseline_result.best_map50_95:.4f}  |  {baseline_result.epochs_completed} epochs  |  {baseline_result.training_time_seconds:.0f}s")
print(f"Augmented -> mAP50: {augmented_result.best_map50:.4f}  |  mAP50-95: {augmented_result.best_map50_95:.4f}  |  {augmented_result.epochs_completed} epochs  |  {augmented_result.training_time_seconds:.0f}s")
print(f"Delta     -> mAP50: {'+' if delta_map50 >= 0 else ''}{delta_map50:.4f}  |  mAP50-95: {'+' if delta_map50_95 >= 0 else ''}{delta_map50_95:.4f}")

In [ ]:
# ── Final visual dashboard ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Dataset size comparison
labels = ["Original", "Synthetic", "Merged"]
img_counts = [stats.total_images, after_stats.total_images, n_orig + n_synth]
x = np.arange(len(labels))
bars = axes[0].bar(x, img_counts, color=["#3498db", "#e74c3c", "#2ecc71"], edgecolor="white")
for bar, cnt in zip(bars, img_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 str(cnt), ha="center", fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_title("Training Images", fontweight="bold")
axes[0].set_ylabel("Count")

# Panel 2: Uniformity + model quality
metric_labels = ["Bucket\nUniform.", "Spatial\nUniform.", "mAP50", "mAP50-95"]
before_vals = [stats.bucket_uniformity, stats.region_uniformity,
               baseline_result.best_map50, baseline_result.best_map50_95]
after_vals = [after_stats.bucket_uniformity, after_stats.region_uniformity,
              augmented_result.best_map50, augmented_result.best_map50_95]
x = np.arange(len(metric_labels))
w = 0.3
axes[1].bar(x - w/2, before_vals, w, label="Baseline / Original", color="#95a5a6")
axes[1].bar(x + w/2, after_vals, w, label="Augmented / Synthetic", color="#2ecc71")
for i, (bv, av) in enumerate(zip(before_vals, after_vals)):
    axes[1].text(i - w/2, bv + 0.015, f"{bv:.3f}", ha="center", fontsize=8)
    axes[1].text(i + w/2, av + 0.015, f"{av:.3f}", ha="center", fontsize=8)
axes[1].set_xticks(x); axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(0, 1.15)
axes[1].set_title("Quality Metrics (higher = better)", fontweight="bold")
axes[1].legend(fontsize=8)

# Panel 3: Summary text
delta_50 = augmented_result.best_map50 - baseline_result.best_map50
delta_95 = augmented_result.best_map50_95 - baseline_result.best_map50_95
info_text = (
    f"Generation\n"
    f"  Methods: {', '.join(result.methods_used)}\n"
    f"  API cost: ${result.total_cost_usd:.2f}\n"
    f"  Synthetic images: {result.total_records}\n"
    f"  Validation: {'PASSED' if result.validation_report and result.validation_report.is_valid else 'N/A'}\n"
    f"\n"
    f"Training ({TRAIN_EPOCHS} epochs, early stop)\n"
    f"  Baseline mAP50:  {baseline_result.best_map50:.4f}\n"
    f"  Augmented mAP50: {augmented_result.best_map50:.4f}\n"
    f"  Delta:           {'+' if delta_50 >= 0 else ''}{delta_50:.4f}\n"
)
axes[2].text(0.05, 0.5, info_text, transform=axes[2].transAxes,
             fontsize=10, verticalalignment="center", fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1", edgecolor="#bdc3c7"))
axes[2].axis("off")
axes[2].set_title("Pipeline Summary", fontweight="bold")

plt.tight_layout()
plt.show()

---
## Step 9: Failure Case Comparison — Baseline vs Augmented

The ultimate test: find validation images where the **baseline model fails** (misses ground-truth detections), then check if the **augmented model** corrects those failures.

For each image we show three columns:
- **Ground Truth** — human annotations
- **Baseline** — predictions from the model trained on original data only
- **Augmented** — predictions from the model trained on original + synthetic data

Missed detections (no prediction with IoU >= 0.5) are counted in each title.

In [ ]:
baseline_result.best_weights.exists()

In [ ]:
from ultralytics import YOLO
from synthdet.types import BBox, AnnotationSource
from pathlib import Path
baseline_result.best_weights = Path("/home/luca/SD4OD/runs/detect/demo_pipeline_output/runs/baseline/weights/best.pt")
augmented_result.best_weights = Path("/home/luca/SD4OD/runs/detect/demo_pipeline_output/runs/augmented/weights/best.pt")

# ── Load both trained models ─────────────────────────────────────────
baseline_model = YOLO(baseline_result.best_weights)
augmented_model = YOLO(augmented_result.best_weights)

CONF_THRESH = 0.25
IOU_THRESH = 0.5

# ── Helpers ──────────────────────────────────────────────────────────
def _iou(a, b):
    """IoU between two (x1, y1, x2, y2) boxes."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

def _to_xyxy(bbox, w, h):
    """Normalized YOLO bbox → pixel (x1, y1, x2, y2)."""
    return (
        (bbox.x_center - bbox.width / 2) * w,
        (bbox.y_center - bbox.height / 2) * h,
        (bbox.x_center + bbox.width / 2) * w,
        (bbox.y_center + bbox.height / 2) * h,
    )

def _count_misses(gt_boxes_xyxy, pred_boxes_xyxy, iou_thresh=0.5):
    """Count GT boxes not matched by any prediction (greedy matching)."""
    matched = set()
    hits = 0
    for gt in gt_boxes_xyxy:
        best_iou, best_pi = 0, -1
        for pi, pred in enumerate(pred_boxes_xyxy):
            if pi in matched:
                continue
            iou = _iou(gt, pred)
            if iou > best_iou:
                best_iou = iou
                best_pi = pi
        if best_iou >= iou_thresh and best_pi >= 0:
            matched.add(best_pi)
            hits += 1
    return len(gt_boxes_xyxy) - hits

def _results_to_bboxes(results, w, h):
    """Convert ultralytics results → list of (BBox, confidence)."""
    out = []
    if results and results[0].boxes is not None and len(results[0].boxes):
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0].cpu().item())
            conf = float(box.conf[0].cpu().item())
            out.append((
                BBox(
                    class_id=cls_id,
                    x_center=((x1 + x2) / 2) / w,
                    y_center=((y1 + y2) / 2) / h,
                    width=(x2 - x1) / w,
                    height=(y2 - y1) / h,
                    source=AnnotationSource.human,
                ),
                conf,
            ))
    return out

def show_preds(img_rgb, pred_tuples, class_names, ax, title):
    """Like show_bboxes but overlays confidence on each box."""
    h, w = img_rgb.shape[:2]
    ax.imshow(img_rgb)
    for bbox, conf in pred_tuples:
        x1 = (bbox.x_center - bbox.width / 2) * w
        y1 = (bbox.y_center - bbox.height / 2) * h
        bw, bh = bbox.width * w, bbox.height * h
        color = COLORS[bbox.class_id % len(COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), bw, bh, linewidth=2,
            edgecolor=color, facecolor="none",
            boxstyle="round,pad=0",
        )
        ax.add_patch(rect)
        label = class_names[bbox.class_id] if bbox.class_id < len(class_names) else str(bbox.class_id)
        ax.text(
            x1, y1 - 4, f"{label} {conf:.0%}", fontsize=6, color="white",
            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"),
        )
    ax.axis("off")
    ax.set_title(title, fontsize=10)

# ── Score every validation image by baseline misses ──────────────────
val_records = dataset.valid
failure_scores = []  # (index, missed, total_gt)

for idx, rec in enumerate(val_records):
    if not rec.bboxes:
        continue
    img = rec.load_image()
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]

    results = baseline_model.predict(img, imgsz=MAX_IMAGE_SIZE, conf=CONF_THRESH, verbose=False)
    preds = _results_to_bboxes(results, w, h)
    pred_xyxy = [_to_xyxy(b, w, h) for b, _ in preds]

    missed = _count_misses(gt_xyxy, pred_xyxy, IOU_THRESH)
    failure_scores.append((idx, missed, len(rec.bboxes)))
    rec.image = None

# Sort by most misses, then most GT boxes
failure_scores.sort(key=lambda x: (-x[1], -x[2]))
top_failures = failure_scores[:4]

print("Validation images where baseline fails most:")
for idx, missed, total in top_failures:
    print(f"  val[{idx}]: {missed}/{total} ground truths missed")

In [ ]:
def resize_if_needed(img, max_size):
    h, w = img.shape[:2]
    if max(h, w) <= max_size:
        return img
    scale = max_size / max(h, w)
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

# ── Side-by-side: Ground Truth | Baseline | Augmented ────────────────
n_imgs = len(top_failures)
fig, axes = plt.subplots(n_imgs, 3, figsize=(18, 5 * n_imgs))
if n_imgs == 1:
    axes = axes[np.newaxis, :]

for row, (idx, baseline_missed, total_gt) in enumerate(top_failures):
    rec = val_records[idx]
    img = resize_if_needed(rec.load_image(), MAX_IMAGE_SIZE)
    img_rgb = to_rgb(img)
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]

    # Column 1: Ground Truth
    show_bboxes(img_rgb, rec.bboxes, dataset.class_names,
                ax=axes[row, 0],
                title=f"Ground Truth ({total_gt} boxes)")

    # Column 2: Baseline predictions
    base_results = baseline_model.predict(img, imgsz=MAX_IMAGE_SIZE,
                                          conf=CONF_THRESH, verbose=False)
    base_preds = _results_to_bboxes(base_results, w, h)
    show_preds(img_rgb, base_preds, dataset.class_names,
               ax=axes[row, 1],
               title=f"Baseline ({len(base_preds)} preds, {baseline_missed} missed)")

    # Column 3: Augmented predictions
    aug_results = augmented_model.predict(img, imgsz=MAX_IMAGE_SIZE,
                                          conf=CONF_THRESH, verbose=False)
    aug_preds = _results_to_bboxes(aug_results, w, h)
    aug_pred_xyxy = [_to_xyxy(b, w, h) for b, _ in aug_preds]
    aug_missed = _count_misses(gt_xyxy, aug_pred_xyxy, IOU_THRESH)
    show_preds(img_rgb, aug_preds, dataset.class_names,
               ax=axes[row, 2],
               title=f"Augmented ({len(aug_preds)} preds, {aug_missed} missed)")

    rec.image = None

# Column headers
for col, label in enumerate(["Ground Truth", "Baseline Model", "Augmented Model"]):
    axes[0, col].text(0.5, 1.12, label, transform=axes[0, col].transAxes,
                      ha="center", fontsize=12, fontweight="bold")

fig.suptitle("Baseline vs Augmented on Hardest Validation Images",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Summary ──────────────────────────────────────────────────────────
total_baseline_missed = sum(m for _, m, _ in top_failures)
total_gt = sum(t for _, _, t in top_failures)

# Recompute augmented misses for the same images
total_aug_missed = 0
for idx, _, _ in top_failures:
    rec = val_records[idx]
    img = rec.load_image()
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]
    aug_results = augmented_model.predict(img, imgsz=MAX_IMAGE_SIZE,
                                          conf=CONF_THRESH, verbose=False)
    aug_preds = _results_to_bboxes(aug_results, w, h)
    aug_pred_xyxy = [_to_xyxy(b, w, h) for b, _ in aug_preds]
    total_aug_missed += _count_misses(gt_xyxy, aug_pred_xyxy, IOU_THRESH)
    rec.image = None

print(f"\nOn the 4 hardest images ({total_gt} ground truth boxes):")
print(f"  Baseline missed:  {total_baseline_missed}/{total_gt}")
print(f"  Augmented missed: {total_aug_missed}/{total_gt}")
improvement = total_baseline_missed - total_aug_missed
if improvement > 0:
    print(f"  Augmented recovered {improvement} detection(s) the baseline missed")
elif improvement == 0:
    print(f"  Both models missed the same detections")
else:
    print(f"  Baseline performed better on these images by {-improvement} detection(s)")

In [ ]:
print(1)